# BioProject (and BioStudies) Metadata validated and modified process (bulk RNA-seq)

&nbsp;

- ヒトの熱ストレスに関連するRNA-Seqにおいて, GEOには登録されていないが、BioProjectに登録されているデータのメタデータを収集,判断する
- 以下のようなクエリで検索 (2026-05-02 update)
- `Search ((((((((((heat stress) OR heat shock) OR high temperature stress) OR hyperthermia) OR heat stroke)) AND "homo sapiens"[Organism]) AND "transcriptome gene expression"[Filter]) AND "bioproject sra"[Filter])) NOT "bioproject gds"[Filter]`
- ID-list: [../../Data/curation/accession_idlist/idlist_bioproject/bioproject_result_260502.txt](../../Data/curation/accession_idlist/idlist_bioproject/bioproject_result_260502.txt)
- 他のクエリでの結果や、manual curation での結果もこちらに統合 (2026-05-03)
- Gene Expression Omnibus (GEO) の PEP format で得られるメタデータにのみあるカラム `sample_molecule_ch1` (Total RNA などの情報) や、`genotype` の情報もマニュアルで追加

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

#### PRJNA1000718 (SINGLE) (RNA-seq)

- validate date 2025-05-18
- [Molecular choreography to acute heat exposure in a simulated extreme high temperature environment](https://doi.org/10.21203/rs.3.rs-3615487/v1)
- おそらくResearch Square 由来のデータ
- 使えそうだな... と思ったが、メタデータに詳細な実験条件と、どのデータがどの条件に合致するか不明なので今回は収集しない

In [4]:
PRJNA1000718 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJNA1000718_metadata.tsv",
    separator="\t"
)

# display(PRJNA1000718)

&nbsp;

#### PRJDB12381 (DRA012808) (SINGLE) (RNA-seq)

- validate date: 2026-05-02
- [Remarkable improvement in detection of readthrough downstream-of-gene transcripts by semi-extractable RNA-sequencing](https://doi.org/10.1261/rna.079469.122)
- Book (Japanese): [“情報”から“マテリアル”へ　ノンコーディングRNA研究](https://www.yodosha.co.jp/yodobook/book/9784758104210/)
- 論文中では `DRA012808` というアクセッション番号に記載がないが、おそらくこの論文に該当?
- コントロールデータが登録されていなかったため、今回は泣く泣く収集しない

In [5]:
# PRJDB12381 = pl.read_csv(
#     "../../Data/metadata/metadata_bioproject/PRJDB12381.metadata.tsv",
#     separator='\t'
# ).filter(
#     pl.col("sample_description").str.contains("Heat stressed ") # heat-stress condition data is performed on HAP1 cells
# )

# PRJDB12381_filtered = PRJDB12381[[s.name for s in PRJDB12381 if not (s.null_count() == PRJDB12381.height)]]

# pl.Config.set_tbl_rows(8)
# display(PRJDB12381_filtered)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# 1． PRJDB12380 (DRA012807) (RNA-seq) (SINGLE)

- ✅ DOWNLOAD (yone-hdd-fantom6)
- ✅ TRIMMING (yone-hdd-fantom6)
- ✅ QUANTIFICATION. (yone-hdd-fantom6)

&nbsp;

- Date of data discovery: 2026-03-14
- validate date: 2026-05-02
- [Remarkable improvement in detection of readthrough downstream-of-gene transcripts by semi-extractable RNA-sequencing](https://doi.org/10.1261/rna.079469.122)
- Book (Japanese): [“情報”から“マテリアル”へ　ノンコーディングRNA研究](https://www.yodosha.co.jp/yodobook/book/9784758104210/)
- `MATERIALS AND METHODS;Cell culture and induction of osmotic stress; ... To induce heat stress (HS), HAP1 cells were exposed at 42°C for 2 h. For aliphatic alcohol treatment, the indicated concentrations of 1,6-hexanediol dissolved in culture medium were added to cells post-OS treatment, followed by incubation for 5 min at room temperature. ...`

In [6]:
PRJDB12380 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB12380.metadata.tsv", # metadata from iseq
    separator='\t'
).filter(
    pl.col("sample_description").str.contains("HAP1") # heat-stress condition data is performed on HAP1 cells
).sort(
    "run_accession",
    descending=False
).with_columns(
    pl.lit("TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)").alias("rna_library"), # please see "MATERIALS AND METHODS" section in the reference paper
    pl.lit("https://doi.org/10.1261/rna.079469.122").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time"),
    pl.lit("37 C").alias("control treatment temperature"),
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "MATERIALS AND METHODS" section in the reference paper
    pl.lit("WT").alias("genotype") 
).drop(
    drop_columns_list
)


PRJDB12380_filtered = PRJDB12380[[s.name for s in PRJDB12380 if not (s.null_count() == PRJDB12380.height)]]

# PRJDB12380_filtered.write_csv(
#     "../Data/metadata/metadata_bioproject/PRJDB12380_modified_metadata.tsv",
#     separator="\t"
# )

# display(PRJDB12380_filtered)

In [7]:
defined_pairs_prjdb12380 = [
    ("DRR321028", "DRR321026"), # Heat stressed and recovered WT HAP1 RNA prepared by conventional method vs. WT HAP1 RNA prepared by conventional method
    ("DRR321029", "DRR321027")  # Heat stressed and recovered WT HAP1 RNA prepared by improved method vs. WT HAP1 RNA prepared by improved method
]

paired_metadata_df_prjdb12380 = create_paired_metadata(
    combined_df=PRJDB12380_filtered,
    defined_sample_pairs=defined_pairs_prjdb12380,
    sample_id_column_name="run_accession"
)

if paired_metadata_df_prjdb12380 is not None:
    paired_metadata_df_prjdb12380.write_csv(
        "../../Data/metadata/metadata_paired/PRJDB12380_paired_metadata.tsv",
        separator="\t"
    )
    display(paired_metadata_df_prjdb12380)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,tag,first_created,experiment_accession,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR321028""","""DRR321026""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407109 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407107""","""SAMD00407109 vs SAMD00407107""","""Homo sapiens""","""DRA012807""","""PolyA""","""2022-12-03""","""593541539 vs 695433398""","""998331814 vs 1204959675""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""Heat stressed and recovered WT HAP1 RNA prepared by conventional method vs WT HAP1 RNA prepared by conventional method""","""DRP009294""","""48176699 vs 60317874""","""Searching for stress inducible architectural noncoding RNAs""","""GENERATED_FILE""","""Heat stressed and recovered WT HAP1 RNA prepared by conventional method vs WT HAP1 RNA prepared by conventional method""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407109 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407107""","""1734361164 vs 2171443464""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00407109 vs SAMD00407107""","""DRR321028 vs DRR321026""","""Truseq stranded mRNA Library Prep Kit(Illumina Inc.)""","""DRX310375 vs DRX310373""","""HOKUDAI""","""HAP1""","""WTHAP_HSrec_Conv vs WTHAP_Ctrl_Conv""","""env_tax;env_tax:terrestrial""","""2022-12-02""","""DRX310375 vs DRX310373""","""Homo sapiens""","""9606""","""PRJDB12380""","""DRR321028 vs DRR321026""","""DRS266840 vs DRS266838""","""2022-12-02""","""DRP009294""","""TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)""","""https://doi.org/10.1261/rna.079469.122""","""42 C""","""120 min.""","""37 C""","""total RNA""","""WT"""
"""DRR321029""","""DRR321027""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407110 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407108""","""SAMD00407110 vs SAMD00407108""","""Homo sapiens""","""DRA012807""","""PolyA""","""2022-12-03""","""635880364 vs 963117646""","""1079612575 vs 1691264824""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""Heat stressed and recovered WT HAP1 RNA prepared by improved method vs WT HAP1 RNA prepared by improved method""","""DRP009294""","""51291309 vs 83087719""","""Searching for stress inducible architectural noncoding RNAs""","""GENERATED_FILE""","""Heat stressed and recovered WT HAP1 RNA prepared by improved method vs WT HAP1 RNA prepared by improved method""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407110 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407108""","""1846487124 vs 2991157884""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00407110 vs SAMD00407108""","""DRR321029 vs DRR321027""","""Truseq stranded mRNA Library Prep Kit(Illumina Inc.)""","""DRX310376 vs DRX310374""","""HOKUDAI""","""HAP1""","""WTHAP_HSrec_Impr vs WTHAP_Ctrl_Impr""","""env_tax;env_tax:terrestrial""","""2022-12-02""","""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. PRJDB13489 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-18
- [Lack of Hikeshi activates HSF1 activity under normal conditions and disturbs the heat-shock response](https://doi.org/10.26508/lsa.202101241)
- おそらく上記の論文が元になっている (この論文にアクセッション番号:DRA013986の記載アリ)
- `Hikeshi` タンパク質について調査している論文
- このプロジェクトは [hTERT-RPE1 cell](https://www.atcc.org/products/crl-4000) に関するデータ
- 明確に以下のような記述が論文中にあるため、収集する
- `Cells and cell cultures Hikeshi-KO HeLa and hTERT-RPE1 cells were previously generated with the CRISPR–Cas9 system (Rahman et al, 2017). HeLa cells stably expressing EGFP-NLS-Hsc70 were previously established by transfection of the pEF5/FRT/EGFP-GW/NLS-Hsc70 plasmid (Rahman et al, 2017).`

&nbsp;

### ERROR

- salmonで発現定量をしている際、以下のようなエラーが発生
- 実行コマンド

```
cwltool --debug --outdir ./HS_human_salmon_quant/HS_human_salmon_quant_se_bioproject/ --cachedir ./cwl_cache/ ./Workflow/03_salmon_quant_subworkflow_se.cwl ./config/salmon/salmon_se_prjdb13489.yml
```

```bash
[2025-05-25 04:31:04.338] [jointLog] [warning] salmon was only able to assign 5 fragments to transcripts in the index, but the minimum number of required assigned fragments (--minAssignedFrags) was 10. This could be indicative of a mismatch between the reference and sample, or a very bad sample.  You can change the --minAssignedFrags parameter to force salmon to quantify with fewer assigned fragments (must have at least 1).
```

- **緊急的に `--minAssignedFrags` を加え、値を1に設定 (デフォルトは10) した `Tools/05_salmon_quant_se_assigncluster.cwl`を作成して実行*  したものの、`processed 16500001 fragments` の部分が異常に早く終わるなど、これまであまり見なかった挙動が見られた
- 一応 発現定量後の情報を見てみたが、ほぼデータがないような変なファイルも入っていた...
- FASTQのファイル取得に失敗したのか？ と思いきや、`iseq`で取得したデータを見てみると、`fastq_bytes` が `307`だったり変なファイルがある...

&nbsp;

- **一旦、このバージョンで定量した結果を残しておき、`--minAssignedFrags` が 10のこれまでのファイルを処理してきたもので処理できないものは外す方針**

In [8]:
PRJDB13489 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB13489_metadata.tsv", # metadata from iseq
    separator='\t'
).with_columns(
    pl.lit("SureSelect Strand-Specific RNA Library Preparation Kits").alias("rna_library"), # please see "Materials and Methods" section in the reference paper
    pl.lit("https://doi.org/10.26508/lsa.202101241").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "Materials and Methods" section in the reference paper
    pl.lit("hTERT-RPE1").alias("cell_line"),

    # add genotype
    pl.when(
        pl.col("sample_title").str.contains("Hikeshi-KO")
    ).then(
        pl.lit("HIKESHI knockout")
    ).otherwise(
        pl.lit("WT")
    ).alias("genotype")
).sort(
    "run_accession",
    descending=False
).drop(
    drop_columns_list
)

PRJDB13489_filtered = PRJDB13489[[s.name for s in PRJDB13489 if not (s.null_count() == PRJDB13489.height)]]

PRJDB13489_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB13489_modified_metadata.tsv",
    separator="\t"
)

# display(PRJDB13489_filtered)

In [9]:
defined_pairs_prjdb13489 = [
    ("DRR361197", "DRR361194"), # hTERT-RPE1 cells under heat-shock condition vs hTERT-RPE1 cells under nonstressed condition
    # ("DRR361200", "DRR361194"), # hTERT-RPE1 cells during recovery for 1.5h from heat stress vs hTERT-RPE1 cells under nonstressed condition
    # ("DRR361203", "DRR361194"), # hTERT-RPE1 cells during recovery for 3h from heat stress vs hTERT-RPE1 cells under nonstressed condition
    ("DRR361206", "DRR361194"), # hTERT-RPE1 cells during recovery for 24h from heat stress vs hTERT-RPE1 cells under nonstressed condition
    ("DRR361198", "DRR361195"), # Hikeshi-KO hTRET-RPE1 cells (R630_1) under heat-shock condition vs Hikeshi-KO hTRET-RPE1 cells (R630_1) under nonstressed condition
    ("DRR361199", "DRR361196"), # Hikeshi-KO hTRET-RPE1 cells (R630_2) under heat-shock condition vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition
    # ("DRR361201", "DRR361195"), # Hikeshi-KO hTRET-RPE1 cells (R630_1) during recovery for 1.5h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_1) under nonstressed condition
    # ("DRR361202", "DRR361196"), # Hikeshi-KO hTRET-RPE1 cells (R630_2) during recovery for 1.5h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition
    ("DRR361204", "DRR361195"), # Hikeshi-KO hTRET-RPE1 cells (R630_1) during recovery for 3h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_1) under nonstressed condition
    ("DRR361205", "DRR361196"), # Hikeshi-KO hTRET-RPE1 cells (R630_2) during recovery for 3h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition
    ("DRR361207", "DRR361195"), # Hikeshi-KO hTRET-RPE1 cells (R630_1) during recovery for 24h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_1) under nonstressed condition
    ("DRR361208", "DRR361196")  # Hikeshi-KO hTRET-RPE1 cells (R630_2) during recovery for 24h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition
]

paired_metadata_df_prjdb13489 = create_paired_metadata(
    combined_df=PRJDB13489_filtered,
    defined_sample_pairs=defined_pairs_prjdb13489,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_prjdb13489 is not None:
    paired_metadata_df_prjdb13489.write_csv(
        "../../Data/metadata/metadata_paired/prjdb13489_paired_metadata.tsv",
        separator='\t'
    )
    
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_prjdb13489)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sex,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,center_name,cell_line,library_name,first_created,experiment_accession,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR361197""","""DRR361194""","""DRR361197 vs DRR361194""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467877 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467874""","""SAMD00467877 vs SAMD00467874""","""Homo sapiens""","""DRA013986""","""PolyA""","""2022-05-11""","""557399607 vs 1020437350""","""791978601 vs 1459468244""","""ILLUMINA""","""female""","""missing""","""Human""","""hTERT-RPE1 cells under heat-shock condition vs hTERT-RPE1 cells under nonstressed condition""","""DRP008510""","""21583328 vs 43462677""","""Gene expression profile of Hikeshi knockout hTERT-RPE1 cell during and after heast shock stress""","""hTERT-RPE1 cells under heat-shock condition vs hTERT-RPE1 cells under nonstressed condition""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467877 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467874""","""776999808 vs 1564656372""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00467877 vs SAMD00467874""","""DRR361197 vs DRR361194""","""SureSelect XT RNA Reagents (Agilent Technologies, Inc)""","""missing""","""DRX347098 vs DRX347095""","""RIKEN_DYNAMICS""","""hTERT-RPE1""","""4_hTERT_RPE1_HS vs 1_hTERT_RPE1_non_HS""","""2022-05-11""","""DRX347098 vs DRX347095""","""PRJDB13489""","""DRS238750 vs DRS238747""","""2022-05-11""","""DRP008510""","""Retina""","""SureSelect Strand-Specific RNA Library Preparation Kits""","""https://doi.org/10.26508/lsa.202101241""","""43 C""","""60 min.""","""37 C (inferred)""","""total RNA""","""WT"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DRR361208""","""DRR361196""","""DRR361208 vs DRR361196""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467888 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467876""","""SAMD00467888 vs SAMD00467876""","""Homo sapiens""","""DRA013986""","""PolyA""","""2022-05-11""","""425834326 vs 769623102""","""600330996 vs 1093175943""","""ILLUMINA""","""female""","""missing""","""Human""","""Hikeshi-KO hTRET-RPE1 cells (R630_2) during recovery for 24h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition""","""DRP008510""","""17107423 vs 31166232""","""Gene expression profile of Hikeshi knockout hTERT-RPE1 cell during and after heast shock stress""","""Hikeshi-KO hTRET-RPE1 cells (R630_2) during recovery for 24h from heat stress vs Hikeshi-KO hTRET-RPE1 cells (R630_2) under nonstressed condition""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467888 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00467876""","""615867228 vs 1121984352""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00467888 vs SAMD00467876""","""DRR361208 vs DRR361196""","""SureSelect XT RNA Reagents (Agilent Technologies, Inc)""","""

&nbsp;

&nbsp;

&nbsp;

## 3. PRJDB3761 (SINGLE) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-05
- [Lack of Hikeshi activates HSF1 activity under normal conditions and disturbs the heat-shock response](https://doi.org/10.26508/lsa.202101241)
- おそらく上記の論文が元になっている (この論文にアクセッション番号:DRA003217の記載アリ)
- `Hikeshi` タンパク質について調査している論文
- 明確に以下のような記述が論文中にあるため、収集する
- `Cells were cultured at 37°C with 5% CO2 in DMEM supplemented with 10% FBS. For heat-shock treatment, cells were incubated for 1 h at 43°C with prewarmed DMEM supplemented with 10% FBS and 20 mM HEPES (pH 7.3) in a water bath.`

In [10]:
PRJDB3761 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB3761_metadata.tsv", # metadata from iseq
    separator='\t'
).with_columns(
    pl.lit("SureSelect Strand-Specific RNA Library Preparation Kits").alias("rna_library"), # please see "Materials and Methods" section in the reference paper
    pl.lit("https://doi.org/10.26508/lsa.202101241").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "Materials and Methods" section in the reference paper
    pl.col("cell_line").fill_null(pl.lit("HeLa")), # fill missing cell_line (all samples are HeLa cells)
    
    # add genotype
    pl.when(
        pl.col("sample_description").str.contains("siRNA")
    ).then(
        pl.lit("HIKESHI knockdown")
    ).when(
        pl.col("sample_description").str.contains("H630")
    ).then(
        pl.lit("HIKESHI knockout")
    ).otherwise(
        pl.lit("WT")
    ).alias("genotype")

).filter(
    ~(pl.col("sample_title").str.contains(r'EGFP-')) # filtered GFP expression data
).sort(
    "run_accession",
    descending=False
).drop(
    drop_columns_list
)

PRJDB3761_filtered = PRJDB3761[[s.name for s in PRJDB3761 if not (s.null_count() == PRJDB3761.height)]]

PRJDB3761_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB3761_modified_metadata.tsv",
    separator="\t"
)

# display(PRJDB3761_filtered)

In [11]:
defined_pairs_prjdb3761 = [
    ("DRR031462", "DRR031460"), # Heat shocked (control siRNA) vs Normal condtion (control siRNA)
    ("DRR031463", "DRR031461"), # Heat shocked (Hikeshi siRNA) vs Normal condtion (Hikeshi siRNA)
    ("DRR031464", "DRR031460"), # Release 1.5hr after heat shock (control siRNA) vs Normal condtion (control siRNA)
    ("DRR031465", "DRR031461"), # Release 1.5hr after heat shock (Hikeshi siRNA) vs Normal condtion (Hikeshi siRNA)
    ("DRR031466", "DRR031460"), # Release 3hr after heat shock (control siRNA) vs Normal condtion (control siRNA)
    ("DRR031467", "DRR031461"), # Release 3hr after heat shock (Hikeshi siRNA) vs Normal condtion (Hikeshi siRNA)
    ("DRR093588", "DRR093585"), # HeLa cells under heat stress vs HeLa cells under normal condition
    ("DRR093591", "DRR093585"), # HeLa cells during recovery for 1.5h from heat stress vs HeLa cells under normal condition
    ("DRR093594", "DRR093585"), # HeLa cells during recovery for 3h from heat stress vs HeLa cells under normal condition
    ("DRR093589", "DRR093586"), # Hikeshi-KO HeLa cells under heat stress vs Hikeshi-KO HeLa cells under normal condition
    ("DRR093590", "DRR093587"), # Hikeshi-KO HeLa cells under heat stress vs Hikeshi-KO HeLa cells under normal condition
    ("DRR093592", "DRR093586"), # Hikeshi-KO HeLa cells during recovery for 1.5h from heat stress vs Hikeshi-KO HeLa cells under normal condition
    ("DRR093593", "DRR093587"), # Hikeshi-KO HeLa cells during recovery for 1.5h from heat stress vs Hikeshi-KO HeLa cells under normal condition
    ("DRR093595", "DRR093586"), # Hikeshi-KO HeLa cells during recovery for 3h from heat stress vs Hikeshi-KO HeLa cells under normal condition
    ("DRR093596", "DRR093587")  # Hikeshi-KO HeLa cells during recovery for 3h from heat stress vs Hikeshi-KO HeLa cells under normal condition
]

paired_metadata_df_prjdb3761 = create_paired_metadata(
    combined_df=PRJDB3761_filtered,
    defined_sample_pairs=defined_pairs_prjdb3761,
    sample_id_column_name="accession" # default value, but specified to clarify
)

# Guard against None so that write_csv is called only on a DataFrame
if paired_metadata_df_prjdb3761 is not None:
    paired_metadata_df_prjdb3761.write_csv(
        "../../Data/metadata/metadata_paired/prjdb3761_paired_metadata.tsv",
        separator="\t"
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_prjdb3761)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,collection_date_end,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,collection_date_start,run_alias,library_construction_protocol,collection_date,experiment_alias,center_name,cell_line,library_name,first_created,strain,experiment_accession,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR031462""","""DRR031460""","""DRR031462 vs DRR031460""","""Illumina HiSeq 2000 sequencing: Illumina HiSeq 2000 sequencing of SAMD00027695 vs Illumina HiSeq 2000 sequencing: Illumina HiSeq 2000 sequencing of SAMD00027693""","""SAMD00027695 vs SAMD00027693""","""Homo sapiens""","""DRA003217""","""cDNA""","""2022-05-11""","""1235569821 vs 1134123752""","""1859667996 vs 1703105533""","""ILLUMINA""","""Generic""","""Heat shocked (control siRNA) vs Normal condtion (control siRNA)""","""DRP008509""","""50133391 vs 46124587""","""Gene expression profile of Hikeshi knockdown HeLa cell during and after heat shock stress""","""2012-10-13""","""HeLa_HS vs HeLa_37""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: Illumina HiSeq 2000 sequencing of SAMD00027695 vs Illumina HiSeq 2000 sequencing: Illumina HiSeq 2000 sequencing of SAMD00027693""","""1804802076 vs 1660485132""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00027695 vs SAMD00027693""","""2012-10-12""","""DRR031462 vs DRR031460""","""null""","""2012-10-13""","""DRX028444 vs DRX028442""","""RIKEN_DYNAMICS""","""Biken-HeLa cell""","""3_HS_1h_GL2ctr1 vs 1_GL2ctr1""","""2022-05-11""","""null""","""DRX028444 vs DRX028442""","""PRJDB3761""","""DRS238725 vs DRS238723""","""2022-05-11""","""DRP008509""","""null""","""SureSelect Strand-Specific RNA Library Preparation Kits""","""https://doi.org/10.26508/lsa.202101241""","""43 C""","""60 min.""","""37 C (inferred)""","""total RNA""","""HIKESHI knockdown"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DRR093596""","""DRR093587""","""DRR093596 vs DRR093587""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00081497 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00081488""","""SAMD00081497 vs SAMD00081488""","""Homo sapiens""","""DRA005851""","""PolyA""","""2022-05-11""","""710361993 vs 1615564445""","""1047192977 vs 2385173255""","""ILLUMINA""","""Generic""","""H630-12_R3h vs H630-12_non-HS""","""DRP008509""","""27005217 vs 60051323""","""Gene expression profile of Hikeshi knockdown HeLa cell during and after heat shock stress""","""null""","""Hikeshi-KO HeLa cells during recovery for 3h from heat stress vs Hikeshi-KO HeLa cells under normal condition""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00081497 vs Illumina HiSeq 2500 sequencing: Illumina HiSeq 2500 sequencing of SAMD00081488""","""972187812 vs 2161847628""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00081497 vs SAMD00081488""","""null""","""DRR093596 vs DRR093587""","""The sequence library was prepared according to the Agilent SureSelect Strand Specific RNA Prep Kit protocol.""","""null""","""DRX087094 vs DRX087085""","""RIKEN_DYNAMICS""","""HeLa""","""Hikeshi-KO HeLa cells during recovery for 3h from heat stress vs Hikeshi-KO HeLa cells under normal condition""",""

&nbsp;

&nbsp;

&nbsp;

#### PRJDB9865 (SINGLE) CAGE-seq

- validate date 2025-05-18
- [Heat shock induces the nuclear accumulation of YAP1 via SRC](https://doi.org/10.1016/j.yexcr.2020.112439)
- 処理を[12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

#### PRJDB7415 (SINGLE) NO DOWNLOAD

- validate date 2025-05-18
- validate date 2025-11-19
- [LncRNA‐dependent nuclear stress bodies promote intron retention through SR protein phosphorylation
](https://doi.org/10.15252/embj.2019102729)
- 論文を発見
- 北海道大学 (現 大阪大学) の廣瀬先生 (https://hirose-lab.com/publications/)
- メタデータを見ると全てのサンプルが `heatshock-exposed` であり、コントロール条件のデータはなさそうなので、残念ながら今回は使用しない

In [12]:
# PRJDB7415 = pl.read_csv(
#     "../../Data/metadata/metadata_bioproject/PRJDB7415_metadata.tsv",
#     separator="\t"
# )

# display(PRJDB7415)

#### PRJDB3999 (RNA-seq) NO DOWNLOAD

- validate date: 2026-05-02
- メタデータが `iseq` コマンドで入手できない
- おそらく以下の論文に関連?
- [Identification of a heat-inducible novel nuclear body containing the long noncoding RNA MALAT1](https://doi.org/10.1242/jcs.253559)
- データをうまく入手できない
- https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SAMD00032086&o=acc_s%3Aa
- SRA run selector に登録されていなかったため、今回は収集しない
- [ENA](https://www.ebi.ac.uk/ena/browser/view/PRJDB3999) を確認しても、`No public data is associated with this project` のため、収集しない

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. PRJEB5979 (ERP5426) (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-18
- [Proteotoxic stress reprograms the chromatin landscape of SUMO modification](https://pmc.ncbi.nlm.nih.gov/articles/PMC6707813/)
- [E-MTAB-2349](https://www.ebi.ac.uk/biostudies/ArrayExpress/studies/E-MTAB-2349?query=E-MTAB-2349) のデータ
- BioStudiesでも検索できたデータ
- 熱ストレス時のデータと、コントロールのデータが存在するので収集する

In [13]:
PRJEB5979 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJEB5979_metadata.tsv",
    separator="\t"
).with_columns(
    pl.lit("Illumina protocol").alias("rna_library"), # add rna_library column from article
    pl.lit("https://doi.org/10.1126/scisignal.aaa2213").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "Materials and Methods" section in the reference paper
    pl.lit("WT").alias("genotype") # no genotype information in the reference paper
).drop(
    drop_columns_list
)

PRJEB5979_filtered = PRJEB5979[[s.name for s in PRJEB5979 if not (s.null_count() == PRJEB5979.height)]]

PRJEB5979_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJEB5979_modified_metadata.tsv",
    separator="\t"
)

# display(PRJEB5979_filtered)

In [14]:
defined_pairs_prjeb5979 = [
    ("ERR468937", "ERR468934"), # E-MTAB-2349:HS_1 vs E-MTAB-2349:UT_1 (UT: untreated)
    ("ERR468932", "ERR468935"), # E-MTAB-2349:HS_2 vs E-MTAB-2349:UT_2 (UT: untreated)
    ("ERR468933", "ERR468936")  # E-MTAB-2349:HS_3 vs E-MTAB-2349:UT_3 (UT: untreated)
]

paired_metadata_df_prjeb5979 = create_paired_metadata(
    combined_df=PRJEB5979_filtered,
    defined_sample_pairs=defined_pairs_prjeb5979,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_prjeb5979 is not None:
    paired_metadata_df_prjeb5979.write_csv(
        "../../Data/metadata/metadata_paired/prjeb5979_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_prjeb5979)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,submitted_galaxy,sra_bytes,submitted_ftp,fastq_bytes,instrument_platform,submitted_aspera,sex,submitted_md5,checklist,submitted_bytes,sample_description,secondary_study_accession,read_count,study_title,sample_title,instrument_model,description,nominal_sdev,cell_type,broker_name,base_count,library_strategy,library_source,library_layout,datahub,nominal_length,status,sample_alias,run_alias,submitted_read_type,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,first_created,experiment_accession,study_accession,submitted_format,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR468937""","""ERR468934""","""ERR468937 vs ERR468934""","""Illumina HiSeq 2000 paired end sequencing: heat shock SUMO-2 RNA-seq""","""SAMEA2422485 vs SAMEA2422483""","""heat shock SUMO-2 RNA-seq""","""ERA295640""","""RANDOM""","""2018-11-16""","""ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468937/HS_1_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468937/HS_1_2.fq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468934/UT_1_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run…","""3357485372 vs 3241974981""","""ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468937/HS_1_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468937/HS_1_2.fq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR468/ERR468934/UT_1_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run…","""2201518752;2180028703 vs 2122204988;2116503599""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/run/ERR468/ERR468937/HS_1_1.fq.gz;fasp.sra.ebi.ac.uk:/vol1/run/ERR468/ERR468937/HS_1_2.fq.gz vs fasp.sra.ebi.ac.uk:/vol1/run/ERR468/ERR468934/UT_1_1.fq.gz;fasp.sra.ebi.ac.uk:/…","""female""","""255969f8b718d3d3b0ede4657c545033;945641c80a5c219876bda7c4b142716b vs 2bfeceece104348825dbd7907b2522c8;684cf210f37fb90b2544ccb27c4d84ce""","""ERC000011""","""2136190004;2114883272 vs 2060970122;2055547032""","""Protocols: Human osteosarcoma (U2OS) cells were obtained from CRUK and grown at 37C according to standard cell culture protocols. U2OS cells were grown at 37C and either left untreated or incubated at…","""ERP005426""","""27482940 vs 26255861""","""heat shock SUMO-2 RNA-seq""","""growing_U2OS_cells_4 vs growing_U2OS_cells_1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 paired end sequencing: heat shock SUMO-2 RNA-seq""","""20""","""human osteosarcoma cell""","""ArrayExpress""","""4946929200 vs 4726054980""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""dcc_arrayexpress""","""200""","""public""","""E-MTAB-2349:growing_U2OS_cells_4 vs E-MTAB-2349:growing_U2OS_cells_1""","""E-MTAB-2349:HS_1 vs E-MTAB-2349:UT_1""","""PAIRED;PAIRED""","""Human osteosarcoma (U2OS) cells were obtained from CRUK and grown at 37C according to standard cell culture protocols. U2OS cells were grown at 37C and either left untreated or incubated at 43C for 4 …","""E-MTAB-2349:12053028 vs E-MTAB-2349:12053025""","""European Bioinformatics Institute;Centre for Gene Regulation and Expression;University of Dundee;Dundee DD1 5EH;United Kingdom""","""U2OS""","""43-1 vs 37-1""","""2014-03-25""","""ERX435171 vs ERX435169""","""PRJEB5979""","""FASTQ;FASTQ""","""ERS427384 vs ERS427382""","""2015-07-16""","""E-MTAB-2349""","""Illumina protocol""","""https://doi.org/10.1126/scisignal.aaa2213""","""43 C""","""240 min.""","""37 C (inferred)""","""total RNA""","""WT"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ERR468933""","""ERR468936""","""ERR468933 vs ERR468936""","""Illumina HiSeq 2000 paired end sequencing:

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# BioStudies

- GEOに登録されていない、かつBioProjectの検索でうまくヒットしないものをここから探索
- 以下のリンクで検索
- [BioStudies search result](https://www.ebi.ac.uk/biostudies/ArrayExpress/studies?facet.study_type=rna-seq+of+coding+rna&facet.study_type=rna-seq+of+non+coding+rna&facet.study_type=rna-seq+of+total+rna&facet.experimental_design=n%2Fa&facet.experimental_design=stimulus+or+stress+design&facet.organism=homo+sapiens&facet.technology=sequencing+assay&facet.assay_by_molecule=rna+assay&facet.raw=true&facet.link_type=biostudies&facet.link_type=ena&query=heat+shock&sortBy=release_date&sortOrder=descending)
- 以下の2つのファイルは`iseq`でダウンロードしたが、うまくダウンロードできていない (2025-05-25)
- [ENAのホームページ: https://www.ebi.ac.uk/ena/browser/view/PRJEB44754](https://www.ebi.ac.uk/ena/browser/view/PRJEB44754)から直接ダウンロード (2025-12-30)

&nbsp;

## 5. PRJEB44754 (ERP128835) (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-18
- validate date: 2025-12-31
- validate date: 2026-05-03
- BioProject の以下のようなクエリでヒットすることを確認 (2026-05-03)
- `((((((("heat stress"[All Fields] OR "heat shock"[All Fields]) OR "high temperature stress"[All Fields]) OR "hyperthermia"[All Fields]) OR "heat stroke"[All Fields]) AND "homo sapiens"[Organism]) AND "bioproject sra"[Filter]) AND "other"[Project Data Type]) NOT "bioproject gds"[Filter]`

&nbsp;

- [Loss of regulation of protein synthesis and turnover underpins an attenuated stress response in senescent human mesenchymal stem cells](https://doi.org/10.1073/pnas.2210745120)
- おそらく上記の論文由来 (筆頭著者、論文内に `E-MTAB-10415` の記載あり)
- `Comparisons were made between donor-matched proliferating and senescent primary hMSCs, under control conditions and when subjected to a 2-h heat shock treatment at 42 °C and following a 24-h recovery period.`
- iseqから取得したメタデータに詳細な実験条件の記載がなかったため、ENAのデータの`BioSample`を検索し、`stimulus` 欄を探しペアを作成する
- ENA browser の `Stimulus`, `Individual`, `Passage`の3つの項目からペアを作成する

&nbsp;

- ENAから直接ダウンロード (2025-12-31)
- このプロジェクトのうち、`ERR5881990`, `ERR5882002`, `ERR5881993`, `ERR5881995`, `ERR5881994`, `ERR5882005` は例えば以下のようなエラーが出たので`fastp`の処理を飛ばした

&nbsp;

```bash
WARNING: The requested image's platform (linux/amd64) does not match the detected host platform (linux/arm64/v8) and no specific platform was requested
[22:01:42] start to load data of read1 
[22:01:42] start to load data of read2 
[22:01:52] Read1: loaded 1M reads 
[22:01:52] Read2: loaded 1M reads 
[22:02:02] Read1: loaded 2M reads 
[22:02:02] Read2: loaded 2M reads 
[22:02:13] Read2: loaded 3M reads 
[22:02:13] Read1: loaded 3M reads 
[22:02:24] Read1: loaded 4M reads 
[22:02:24] Read2: loaded 4M reads 
[22:02:35] Read2: loaded 5M reads 
[22:02:35] Read1: loaded 5M reads 
ERROR: sequence and quality have different length:
@ERR5881990.5749217 K00311:137:HKNFLBBXY:6:1207:28483:306JJJJJJJ1C1:137:H9JJJJFFJF5JJJJJJGGCA2
TTTTCA7<F-FJ<<7FAJ--F-7--<7--7--77-77-ATGGAA
+
31382TAAGGTTATTATTGGAAACATTACTTTGAGTGCAGAJJFJ<AJAAJJJFFFFFJFAJFJF16/2
ERROR: sequence and quality have different length
```

&nbsp;

In [15]:
PRJEB44754 = pl.read_csv(
    "../../Data/metadata/metadata_biostudies/ERP128835.metadata.tsv",
    separator="\t"
).with_columns(
    pl.lit("TruSeq Stranded mRNA assay").alias("rna_library"), # add rna_library column from ENA webpage
    pl.lit("https://doi.org/10.1073/pnas.2210745120").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("polyA RNA").alias("sample_molecule_ch1"), # please see "Materials and Methods" section in the reference paper
    pl.lit("WT").alias("genotype") # no genotype information in the reference paper
).drop(
    drop_columns_list
)

PRJEB44754_filtered = PRJEB44754[[s.name for s in PRJEB44754 if not (s.null_count() == PRJEB44754.height)]]

PRJEB44754_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJEB44754_modified_metadata.tsv",
    separator="\t"
)

# display(PRJEB44754_filtered)

In [16]:
# failed trimming run
# ERR5881990
# ERR5882002
# ERR5881993
# ERR5881995
# ERR5881994
# ERR5882005


defined_pairs_prjeb44754 = [
    # Individual:TH223, Early passage
    # ("ERR5881991", "ERR5881993"), # heat-shock vs none
    # ("ERR5881992", "ERR5881993"), # heat-shock vs none
    # Individual:TH223, Late passage
    ("ERR5882000", "ERR5882007"), # heat-shock vs none
    # Individual:TH270, Early passage
    # ("ERR5881989", "ERR5881995"), # heat-shock vs none
    # Individual:TH270, Late passage
    ("ERR5881999", "ERR5882003"), # heat-shock vs none
    # Individual:TH296, Early passage data is None
    # Individual:TH296, Late passage data
    ("ERR5882001", "ERR5882004"), # heat-shock vs none
    # Individual:TH305, Early passage
    # ("ERR5881988", "ERR5881994"), # heat-shock vs none
    # Individual:TH305, Late passage
    ("ERR5881998", "ERR5882006"), # heat-shock vs none
    # Individual:WH176, Early passage
    # ("ERR5881990", "ERR5881996"), # heat-shock vs none
    # # Individual:WH176, Late passage
    # ("ERR5882002", "ERR5882005")  # heat-shock vs none
]

paired_metadata_df_prjeb44754 = create_paired_metadata(
    combined_df=PRJEB44754_filtered,
    defined_sample_pairs=defined_pairs_prjeb44754,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_prjeb44754 is not None:
    paired_metadata_df_prjeb44754.write_csv(
        "../../Data/metadata/metadata_paired/prjeb44754_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_prjeb44754)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,submitted_galaxy,submitted_ftp,fastq_bytes,instrument_platform,submitted_aspera,bam_ftp,submitted_md5,checklist,submitted_bytes,sample_description,secondary_study_accession,read_count,study_title,sample_title,instrument_model,bam_bytes,description,cell_type,broker_name,base_count,library_strategy,library_source,library_layout,dev_stage,bam_aspera,datahub,status,sample_alias,run_alias,submitted_read_type,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,first_created,experiment_accession,scientific_name,tax_id,study_accession,submitted_format,bam_galaxy,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR5882000""","""ERR5882007""","""ERR5882000 vs ERR5882007""","""Illumina HiSeq 4000 sequencing: RNA-seq of early- and late-passage human mesenchymal stem cells (5 donors, donor-matched) with and without treatment with 2 hours of 42 degrees C heat shock""","""SAMEA8691607 vs SAMEA8691614""","""RNA-seq of early- and late-passage human mesenchymal stem cells (5 donors, donor-matched) with and without treatment with 2 hours of 42 degrees C heat shock""","""ERA4206367""","""PolyA""","""2022-04-17""","""ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882000/JL2_S53_R1_001.fastq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882000/JL2_S53_R2_001.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882007/JL5_S56_R1_001…","""ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882000/JL2_S53_R1_001.fastq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882000/JL2_S53_R2_001.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR588/ERR5882007/JL5_S56_R1_001…","""1429587438;1530839711 vs 1455980828;1572644625""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/run/ERR588/ERR5882000/JL2_S53_R1_001.fastq.gz;fasp.sra.ebi.ac.uk:/vol1/run/ERR588/ERR5882000/JL2_S53_R2_001.fastq.gz vs fasp.sra.ebi.ac.uk:/vol1/run/ERR588/ERR5882007/JL5_S56_…","""ftp.sra.ebi.ac.uk/vol1/err/ERR588/000/ERR5882000.bam vs ftp.sra.ebi.ac.uk/vol1/err/ERR588/007/ERR5882007.bam""","""b0764578ec8b7f74282c82724d1796cf;3abb679bed5cf3fe28f23ab6ef4eca8d vs 516058d933c0b581fefc11a024e368c2;1b994e22248f46c2d4934eccb38d249b""","""ERC000011""","""1482303935;1587138860 vs 1506159460;1627098414""","""Protocols: Early and late passage human mesenchymal stem cells were heat shocked (42°C, 2 hour, 5% CO2) and compared with controls kept at 37°C, 5% CO2. Following treatment, cells in culture flasks we…","""ERP128835""","""66298156 vs 66308310""","""RNA-seq of early- and late-passage human mesenchymal stem cells (5 donors, donor-matched) with and without treatment with 2 hours of 42 degrees C heat shock""","""Sample 2 vs Sample 5""","""Illumina HiSeq 4000""","""2798177226 vs 2861522922""","""Illumina HiSeq 4000 sequencing: RNA-seq of early- and late-passage human mesenchymal stem cells (5 donors, donor-matched) with and without treatment with 2 hours of 42 degrees C heat shock""","""mesenchymal stem cell of the bone marrow""","""ArrayExpress""","""4937958348 vs 4979466700""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""adult""","""fasp.sra.ebi.ac.uk:/vol1/err/ERR588/000/ERR5882000.bam vs fasp.sra.ebi.ac.uk:/vol1/err/ERR588/007/ERR5882007.bam""","""dcc_arrayexpress""","""public""","""E-MTAB-10415:Sample 2 vs E-MTAB-10415:Sample 5""","""E-MTAB-10415:JL2_S53_R vs E-MTAB-10415:JL5_S56_R""","""PAIRED;PAIRED""","""Early and late passage human mesenchymal stem cells were heat shocked (42°C, 2 hour, 5% CO2) and compared with controls kept at 37°C, 5% CO2. Following treatment, cells in culture fl

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### PRJEB73803 (ERP158539) (PAIRED) 4sU-seq

- validate date: 2025-05-18
- [E-MTAB-13903](https://www.ebi.ac.uk/biostudies/ArrayExpress/studies/E-MTAB-13903?query=E-MTAB-13903)
- 論文は見つけられなかったが、熱ショックとコントロールのデータが登録されているので念の為収集する
- `control exposed to 200nM 4sU` と `HS-treated exposed to 200nM 4sU` の2つがある
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

## BioProject Manual Curation (2026~)

- マニュアルで収集した BioProject のデータ (GEOにはない)

&nbsp;

## 6. PRJEB64703 (RNA-seq)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- 以下のクエリでヒットしたデータ (2026-05-03)
- `(((((((((("heat stress") OR "heat shock") OR "high temperature stress") OR "hyperthermia") OR "heat stroke")) AND "homo sapiens"[Organism]) AND "bioproject sra"[Filter]) AND "other"[Project Data Type])) NOT "bioproject gds"[Filter]`
- [ENA Browser: PRJEB64703](https://www.ebi.ac.uk/ena/browser/view/PRJEB64703)
- [BioRxiv: The piggyBac derived transposase 5 (PGBD5) can interact with human piggyBac-like elements](https://doi.org/10.1101/2025.07.31.667870)
- 同じ論文由来の ChIP-seq のデータは、熱ストレス条件下ではない
- `For the heat shock, cells were incubated for 3 hours at 42°C and with 5% CO2.`

In [17]:
PRJEB64703 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/PRJEB64703.metadata.tsv",
    separator='\t'
).with_columns(
    pl.lit("NEBNext Ultra II RNA Library Prep Kit for Illumina").alias("rna_library"), # please see "Materials and Methods" section in the reference paper
    pl.lit("https://doi.org/10.1101/2025.07.31.667870").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("180 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "Materials and Methods" section in the reference paper
    pl.lit("WT").alias("genotype"), # no genotype information in the reference paper
    pl.lit("HeLa").alias("cell_line")
).drop(
    drop_columns_list
)

PRJEB64703_filtered = PRJEB64703[[s.name for s in PRJEB64703 if not (s.null_count() == PRJEB64703.height)]]

# pl.Config.set_tbl_rows(8)
# display(PRJEB64703_filtered)

In [18]:
defined_pairs_prjeb64703 = [
    ("ERR11785749", "ERR11785751"), # HeLa-HS1 vs. HeLa-control1
    ("ERR11785750", "ERR11785752")  # HeLa-HS2 vs. HeLa-control2
]

paired_metadata_df_prjeb64703 = create_paired_metadata(
    combined_df=PRJEB64703_filtered,
    defined_sample_pairs=defined_pairs_prjeb64703,
    sample_id_column_name="accession"
)

if paired_metadata_df_prjeb64703 is not None:
    paired_metadata_df_prjeb64703.write_csv(
        "../../Data/metadata/metadata_paired/prjeb64703_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_prjeb64703)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,submitted_galaxy,submitted_ftp,fastq_bytes,instrument_platform,submitted_aspera,bam_ftp,submitted_md5,checklist,submitted_bytes,country,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,collection_date_end,sample_title,instrument_model,bam_bytes,description,bam_file_role,base_count,library_strategy,library_source,library_layout,bam_aspera,status,sample_alias,collection_date_start,run_alias,submitted_file_role,submitted_read_type,collection_date,experiment_alias,center_name,cell_line,library_name,tag,first_created,experiment_accession,scientific_name,tax_id,study_accession,submitted_format,bam_galaxy,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR11785749""","""ERR11785751""","""ERR11785749 vs ERR11785751""","""Illumina HiSeq 4000 sequencing""","""SAMEA114228214 vs SAMEA114228212""","""HeLa cells RNA-seq following stressing treatments""","""ERA26253193""","""cDNA""","""2025-08-06""","""ftp.sra.ebi.ac.uk/vol1/run/ERR117/ERR11785749/HeLa-mRNA-HS.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR117/ERR11785751/HeLa-mRNA-T.fastq.gz""","""ftp.sra.ebi.ac.uk/vol1/run/ERR117/ERR11785749/HeLa-mRNA-HS.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR117/ERR11785751/HeLa-mRNA-T.fastq.gz""","""2880773624 vs 2889832414""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/run/ERR117/ERR11785749/HeLa-mRNA-HS.fastq.gz vs fasp.sra.ebi.ac.uk:/vol1/run/ERR117/ERR11785751/HeLa-mRNA-T.fastq.gz""","""ftp.sra.ebi.ac.uk/vol1/err/ERR117/049/ERR11785749.bam vs ftp.sra.ebi.ac.uk/vol1/err/ERR117/051/ERR11785751.bam""","""bec86b2950d10424635cd480460783ac vs bbffd58fe46f2d1cedadfa2772edd978""","""ERC000011""","""2695075030 vs 2702024757""","""France""","""HeLa-HS1-RNAseq vs HeLa-control1-RNAseq""","""ERP149894""","""57642285 vs 58301251""","""HeLa cells RNA-seq following heat shock, geldanamycin and radicicol treatments""","""GENERATED_FILE""","""2016-12-31""","""HeLa-HS1 vs HeLa-control1""","""Illumina HiSeq 4000""","""2766500012 vs 2777452203""","""Illumina HiSeq 4000 sequencing""","""ARCHIVAL_FILE""","""2921664610 vs 2958982679""","""ssRNA-seq""","""TRANSCRIPTOMIC""","""SINGLE""","""fasp.sra.ebi.ac.uk:/vol1/err/ERR117/049/ERR11785749.bam vs fasp.sra.ebi.ac.uk:/vol1/err/ERR117/051/ERR11785751.bam""","""public""","""HeLa-HS1 vs HeLa-control1""","""2016-01-01""","""ena-RUN-TAB-02-08-2023-10:11:57:062-8360 vs ena-RUN-TAB-02-08-2023-10:11:57:063-8364""","""SUBMISSION_FILE""","""SINGLE""","""2016""","""ena-EXPERIMENT-TAB-02-08-2023-10:11:57:062-8359 vs ena-EXPERIMENT-TAB-02-08-2023-10:11:57:062-8363""","""INRA""","""HeLa""","""HeLa-HS1 vs HeLa-control1""","""env_tax:terrestrial;env_tax""","""2023-08-02""","""ERX11184266 vs ERX11184268""","""Homo sapiens""","""9606""","""PRJEB64703""","""FASTQ""","""ftp.sra.ebi.ac.uk/vol1/err/ERR117/049/ERR11785749.bam vs ftp.sra.ebi.ac.uk/vol1/err/ERR117/051/ERR11785751.bam""","""ERS16211670 vs ERS16211668""","""2025-08-06""","""df50ce6c-775c-40fd-81b0-da5cde624e13""","""NEBNext Ultra II RNA Library Prep Kit for Illumina""","""https://doi.org/10.1101/2025.07.31.667870""","""42 C""","""180 min.""","""37 C (inferred)""","""total RNA""","""WT"""
"""ERR11785750""","""ERR11785752""","""ERR11785750 vs ERR11785752""","""Illumina HiSeq 4000 sequencing""","""SAMEA114228215 vs SAMEA114228213""","""HeLa cells RNA-seq following stressing treatments""","""ERA26253193""","""cDNA""","""2025-08-06""","""ftp.sra.ebi.ac.uk/vol1/run/ERR117/ERR11785750/HS.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/run/E

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7. PRJDB9451 (DRA009793) (RNA-seq)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- Date of data discovery: 2026-03-14
- validate date: 2026-05-02

&nbsp;

- [Remarkable improvement in detection of readthrough downstream-of-gene transcripts by semi-extractable RNA-sequencing](https://doi.org/10.1261/rna.079469.122)
- このデータは、BioProject の検索でもヒットしなかった
- Book (Japanese): [“情報”から“マテリアル”へ　ノンコーディングRNA研究](https://www.yodosha.co.jp/yodobook/book/9784758104210/)
- `MATERIALS AND METHODS;Cell culture and induction of osmotic stress; ... To induce heat stress (HS), HAP1 cells were exposed at 42°C for 2 h. For aliphatic alcohol treatment, the indicated concentrations of 1,6-hexanediol dissolved in culture medium were added to cells post-OS treatment, followed by incubation for 5 min at room temperature. ...`

In [19]:
PRJDB9451 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB9451.metadata.tsv", # metadata from iseq
    separator='\t'
).filter(
    pl.col("sample_description").str.contains("HAP1") # heat-stress condition data is performed on HAP1 cells
).sort(
    "run_accession",
    descending=False
).drop(
    drop_columns_list
).with_columns(
    pl.lit("TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)").alias("rna_library"), # please see "MATERIALS AND METHODS" section in the reference paper
    pl.lit("https://doi.org/10.1261/rna.079469.122").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time"),
    pl.lit("37 C").alias("control treatment temperature"),
    pl.lit("total RNA").alias("sample_molecule_ch1"), # please see "MATERIALS AND METHODS" section in the reference paper
    pl.lit("WT").alias("genotype") 
)

PRJDB9451_filtered = PRJDB9451[[s.name for s in PRJDB9451 if not (s.null_count() == PRJDB9451.height)]]

# PRJDB9451_filtered.write_csv(
#     "../Data/metadata/metadata_bioproject/PRJDB9451_modified_metadata.tsv",
#     separator="\t"
# )

# pl.Config.set_fmt_str_lengths(200)
# display(PRJDB9451_filtered)

In [20]:
defined_pairs_prjdb9451 = [
    ("DRR214953", "DRR214951"), # HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by conventional TRIregeant protocol vs. HAP1 cells Normal culture condition, Total RNAs were extracted by conventional TRIregeant protocol
    ("DRR214954", "DRR214952")  # HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by improved TRIregeant protocol vs. HAP1 cells Normal culture condition, Total RNAs were extracted by improved TRIregeant protocol
]

paired_metadata_df_prjdb9451 = create_paired_metadata(
    combined_df=PRJDB9451_filtered,
    defined_sample_pairs=defined_pairs_prjdb9451,
    sample_id_column_name="run_accession"
)

if paired_metadata_df_prjdb9451 is not None:
    paired_metadata_df_prjdb9451.write_csv(
        "../../Data/metadata/metadata_paired/PRJDB9451_paired_metadata.tsv",
        separator="\t"
    )
    display(paired_metadata_df_prjdb9451)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,tag,first_created,experiment_accession,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR214953""","""DRR214951""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209816 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209814""","""SAMD00209816 vs SAMD00209814""","""Homo sapiens""","""DRA009793""","""Inverse rRNA""","""2022-12-21 vs 2022-03-11""","""1468198927 vs 1861273855""","""2368491635 vs 3029873213""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by conventional TRIregeant protocol vs HAP1 cells Normal culture condition, Total RNAs were extr…","""DRP008266""","""79583343 vs 100955243""","""Total RNA-seq of stressed cells upon conventional or improved RNA extraction""","""GENERATED_FILE""","""7_2_WTHAP_HSrec_Conv vs 1_2_WTHAP_Ctrl_Conv""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209816 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209814""","""3979167150 vs 5047762150""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00209816 vs SAMD00209814""","""DRR214953 vs DRR214951""","""TruSeq Stranded mRNA HT Sample Prep Kit (skip the polyA selection step)""","""DRX205288 vs DRX205286""","""HOKUDAI""","""HAP1 cell""","""7_2_WTHAP_HSrec_Conv vs 1_2_WTHAP_Ctrl_Conv""","""env_tax;env_tax:terrestrial""","""2022-03-08""","""DRX205288 vs DRX205286""","""Homo sapiens""","""9606""","""PRJDB9451""","""DRR214953 vs DRR214951""","""DRS230493 vs DRS230491""","""2022-03-08""","""DRP008266""","""TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)""","""https://doi.org/10.1261/rna.079469.122""","""42 C""","""120 min.""","""37 C""","""total RNA""","""WT"""
"""DRR214954""","""DRR214952""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209817 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209815""","""SAMD00209817 vs SAMD00209815""","""Homo sapiens""","""DRA009793""","""Inverse rRNA""","""2022-03-11""","""2006387263 vs 1199349638""","""3354762464 vs 1988136569""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by improved TRIregeant protocol vs HAP1 cells Normal culture condition, Total RNAs were extracte…","""DRP008266""","""109242105 vs 64933636""","""Total RNA-seq of stressed cells upon conventional or improved RNA extraction""","""GENERATED_FILE""","""8_2_WTHAP_HSrec_Impr vs 2_2_WTHAP_Ctrl_Impr""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209817 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209815""","""5462105250 vs 3246681800""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00209817 vs SAMD00209815""","""DRR214954 vs DRR214952""","""TruSeq Stranded mRNA HT Sample Prep Kit (skip the polyA selection step)""","""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Concat dataframe (RNA-seq only)

In [21]:
dfs_rna_seq_bioproject_biostudies: list[pl.DataFrame | None] = [
    paired_metadata_df_prjdb12380,
    paired_metadata_df_prjdb13489,
    paired_metadata_df_prjdb3761,
    paired_metadata_df_prjeb5979,
    paired_metadata_df_prjeb44754,
    paired_metadata_df_prjeb64703,
    paired_metadata_df_prjdb9451
]

rna_seq_bioproject_biostudies: list[pl.DataFrame] = [
    df for df in dfs_rna_seq_bioproject_biostudies if df is not None
]

In [22]:
rna_seq_bioproject_biostudies = pl.concat(rna_seq_bioproject_biostudies, how="diagonal_relaxed")
rna_seq_bioproject_biostudies.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_bioproject_biostudies_metadata_{execute_date}.tsv",
    separator="\t"
)